<a href="https://colab.research.google.com/github/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/3_supervised/Hands_on_3_CanonChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *The Digital Librarian* v.2

This session's experiments are largely inspired by the articles: *Literary Canonicity and Algorithmic Fairness: The Effect of Author Gender on Classification Models* [(Lassen et al., 2023)](https://ceur-ws.org/Vol-3834/paper76.pdf); and *Operationalizing Canonicity: A Quantitative Study of French 19th and 20th Century Literature* [(Barré et al., 2023)](https://culturalanalytics.org/article/88113-operationalizing-canonicity-a-quantitative-study-of-french-19th-and-20th-century-literature).

The data used comes from the [ANRChapitres/2000romans19e20e](https://doi.org/10.5281/zenodo.7446728).

## Canonicity Prediction and Fairness


<!-- ![Woman writing. Edouard Manet (c. 1883)](https://uploads7.wikiart.org/images/edouard-manet/woman-writing.jpg!Large.jpg)
<p align="right">
  <i>Woman writing</i>. Edouard Manet (c. 1883). Huile sur toile.
</p> -->

![Empirical Construction, Istanbul. Julie Mehretu (2003)](https://cdn.sanity.io/images/476nwnl9/production/6e85372d72734b07c7fc395fed0d6050cbdd5bb4-3000x2015.jpg)
<p align="right">
  <i>Empirical Construction, Istanbul</i>. Julie Mehretu (2003). Acrylic and ink on canvas.
</p>




# 🟢 Import libraries

In [36]:
# For deep learning
import torch
from torch.utils.data import DataLoader

# For (pre-trained) LMs
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# For data handling
import pandas as pd
from datasets import load_dataset

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# For document representation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer

# Fro general purposes
from tqdm import tqdm
import numpy as np
import random

# For visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github, load_dataset_from_github

--2025-11-10 12:46:52--  https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3469 (3.4K) [text/plain]
Saving to: ‘helpers.py.1’

helpers.py.1        100%[===================>]   3.39K  --.-KB/s    in 0s      

2025-11-10 12:46:52 (28.9 MB/s) - ‘helpers.py.1’ saved [3469/3469]



# 🔵 Helpers

The following section provides a few helper functions for transparency of evaluation and format checking.

## Scoring Computation

Your submissions will be scored for two main criteria, along the following rationale:
- Performance: your model should be able to classify correctly 'canon' and 'non-canon' excerpts
- Fairness: your model should not exhibit systematic differneces in performance (or decision distribution) across groups (here, authors' gender).

These scores are computed with the following methods: `classification_metrics`, and `fairness_module` shown below (feel free to use them during training/validation!). The main criteria that will be examined for the submissions are: the F1-score (considering the 'canon' class as the positive class); and fairness $\epsilon^\star$ (see below for more information), the largest \epsilon value so that all fairness ratios are bounded by: $\epsilon^\star \leq \mathrm{ratio} \leq 1/\epsilon^\star$.

You can use these functions with the following code (adapted to your variables):

- for performance - `classification_metrics`:
```python
validation_df = dataset["validation"].to_pandas()
performance_metrics = classification_metrics(
    y_true=validation_df["label"],                     # true labels
    y_pred=MY_PREDICTIONS,                             # YOUR PREDICTIONS HERE: predictions list
    pos_label=0,                                       # positive class label
)
print(performance_metrics)
```

- for fairness - `fairness_module`:
```python
validation_df = dataset["validation"].to_pandas()
fairness_metrics = fairness_module(
    y_true=validation_df["label"],                     # true labels
    y_pred=MY_PREDICTIONS,                             # YOUR PREDICTIONS HERE: predictions list
    attributes=validation_df["author_gender"],         # list of attributes
    pos_label=0,                                       # positive class label
    protected_attribute=0,                             # protected attributed label
    epsilon=.8,                                        # parameter for 'fair' region
    title="Fairness Check for Binary Classifier",      # title for the potential plot
    plot=True,                                         # plot the results
)
print(fairness_metrics)
```

<details><summary>A few notes about the <i>fairness</i> metrics and interpretation.</summary>

---

### Fairness Indicators and Interpretation

* **Equal Opportunity Ratio (TPR ratio)**

  * **Definition:** Compares the *true positive rate* (TPR = TP / (TP + FN)) between unprivileged and privileged groups.
  * **Interpretation:** Measures how equally the model identifies positive cases across groups.

    * Ratio ≈ 1 → both groups have similar sensitivity.
    * Ratio < 1 → the unprivileged group receives fewer true positives (possible discrimination).

* **Predictive Parity Ratio (PPV ratio)**

  * **Definition:** Compares the *positive predictive value* (PPV = TP / (TP + FP)) between groups.
  * **Interpretation:** Indicates whether positive predictions are equally reliable across groups.

    * Ratio ≈ 1 → similar precision (prediction trustworthiness) for both groups.
    * Ratio < 1 → predictions for the unprivileged group are less reliable.

* **Predictive Equality Ratio (FPR ratio)**

  * **Definition:** Compares *false positive rates* (FPR = FP / (FP + TN)) between groups.
  * **Interpretation:** Evaluates if the model unfairly flags negatives as positives.

    * Ratio ≈ 1 → similar false-alarm rates.
    * Ratio > 1 → unprivileged group experiences more false positives → bias concern.

* **Accuracy Equality Ratio (ACC ratio)**

  * **Definition:** Compares overall *accuracy* ((TP + TN) / (TP + TN + FP + FN)) between groups.
  * **Interpretation:** Reflects whether the model performs equally well across groups.

    * Ratio ≈ 1 → balanced predictive performance.
    * Ratio < 1 → poorer accuracy for the unprivileged group.

* **Statistical Parity Ratio (PR ratio)**

  * **Definition:** Compares the *positive prediction rate* ((TP + FP) / Total) between groups.
  * **Interpretation:** Tests whether both groups have equal likelihood of being predicted as positive.

    * Ratio ≈ 1 → equal selection rates.
    * Ratio < 1 or > 1 → potential disparate impact (under- or over-selection).

---

### General Fairness Criterion

For all metrics, **a ratio in the range**
$
\epsilon < \text{ratio} < \frac{1}{\epsilon}
$
is considered *fair*,
where ($ \epsilon \in [0, 1] $) (with **ε = 1** meaning perfect fairness).
Values outside this interval signal possible bias against the unprivileged group.

As an aggregate score, we additionally compute $\epsilon^\star$: the largest possible value of $\epsilon$ so that all indicatores are within $[\epsilon,\ 1/\epsilon]$. The closer to 1, the fairer the predictor.


</details>

In [45]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# =================================================================
# ------------------------ PERFORMANCE ----------------------------
# =================================================================

def classification_metrics(y_true, y_pred, pos_label=0):
    """
    Computes basic classification metrics: precision, recall, F1-score, and accuracy.

    Parameters
    ----------
    y_true : array-like
        True labels.
    y_pred : array-like
        Predicted labels.
    pos_label : int or str
        Label considered as the positive class.

    Returns
    -------
    metrics_dict : dict
        Dictionary containing precision, recall, f1-score, and accuracy.
    """
    precision = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    metrics_dict = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }

    return metrics_dict

In [46]:
# =================================================================
# --------------------------- FAIRNESS ----------------------------
# =================================================================

def fairness_module(
    y_true,
    y_pred,
    attributes,
    pos_label=0,
    protected_attribute=0,
    epsilon=.8,
    title="Fairness Check for Binary Classifier",
    plot=False,
):
    """
    Computes fairness metrics (as dalex) and plots a dalex-like fairness check.

    Parameters
    ----------
    y_true : array-like
        True labels.
    y_pred : array-like
        Predicted labels.
    pos_label : int or str
        Label of the positive class.
    attributes : array-like
        Protected attribute values per observation.
    protected_attribute : value
        Attribute value considered unprivileged.
    epsilon : float, default=.8
        Threshold defining the "fair zone" (ε ≤ ratio ≤ 1/ε).
    title : str, optional
        Title of the plot.

    Returns
    -------
    fairness_df : pd.DataFrame
        DataFrame with fairness ratios.
    max_epsilon : float
        Largest epsilon so that all ratios lie in [ε, 1/ε].
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    attributes = np.array(attributes)

    unpriv_mask = attributes == protected_attribute
    priv_mask = ~unpriv_mask

    def confusion_vals(y_t, y_p):
        tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[1 - pos_label, pos_label]).ravel()
        return tn, fp, fn, tp

    tn_u, fp_u, fn_u, tp_u = confusion_vals(y_true[unpriv_mask], y_pred[unpriv_mask])
    tn_p, fp_p, fn_p, tp_p = confusion_vals(y_true[priv_mask], y_pred[priv_mask])

    def safe_div(a, b): return a / b if b != 0 else np.nan

    # Unprivileged metrics
    tpr_u = safe_div(tp_u, tp_u + fn_u)
    ppv_u = safe_div(tp_u, tp_u + fp_u)
    fpr_u = safe_div(fp_u, fp_u + tn_u)
    acc_u = safe_div(tp_u + tn_u, tp_u + tn_u + fp_u + fn_u)
    pr_u  = safe_div(tp_u + fp_u, tp_u + tn_u + fp_u + fn_u)

    # Privileged metrics
    tpr_p = safe_div(tp_p, tp_p + fn_p)
    ppv_p = safe_div(tp_p, tp_p + fp_p)
    fpr_p = safe_div(fp_p, fp_p + tn_p)
    acc_p = safe_div(tp_p + tn_p, tp_p + tn_p + fp_p + fn_p)
    pr_p  = safe_div(tp_p + fp_p, tp_p + tn_p + fp_p + fn_p)

    ratios = {
        "Equal opportunity ratio (TPR)": safe_div(tpr_u, tpr_p),
        "Predictive parity ratio (Precision)": safe_div(ppv_u, ppv_p),
        "Predictive equality ratio (FPR)": safe_div(fpr_u, fpr_p),
        "Accuracy equality ratio (ACC)": safe_div(acc_u, acc_p),
        "Statistical parity ratio (PPR)": safe_div(pr_u, pr_p),
    }

    fairness_df = pd.DataFrame(list(ratios.items()), columns=["metric", "ratio"])

    # Compute maximal epsilon such that all ratios are within [ε, 1/ε]
    max_epsilon = min(np.min(fairness_df["ratio"]), 1 / np.max(fairness_df["ratio"]))
    ratios.update({'fairness_epsilon':max_epsilon})

    # ---- Plot ----
    if plot:
        fig, ax = plt.subplots(figsize=(8, 5))
        y_pos = np.arange(len(fairness_df))
        ratios_values = fairness_df["ratio"].values

        # Fairness zones
        ax.axvspan(epsilon, 1/epsilon, color="palegreen", alpha=0.3, label=f"Fair zone [{epsilon:.2f}, {1/epsilon:.2f}]")
        ax.axvspan(0, epsilon, color="mistyrose", alpha=0.5)
        ax.axvspan(1/epsilon, max(1.5, max(ratios_values) + 0.1), color="mistyrose", alpha=0.5)
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1)

        # Bars
        ax.barh(y_pos, ratios_values-1, color="teal", alpha=0.8, left=1) # bars start from 1
        ax.set_yticks(y_pos)
        ax.set_yticklabels(fairness_df["metric"])
        ax.invert_yaxis()  # like dalex: top metric first
        ax.set_xlabel("Unprivileged / Privileged Ratio", fontsize=11)
        ax.set_xlim(0, max(1.5, np.nanmax(ratios_values) + 0.1))
        ax.set_title(title, color="darkblue", fontsize=13)
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    return ratios

## Sanity checks

Quick check functions, to run before submission: make sure your predictions are in the right format!

In [ ]:
# ======================================================================
# Some security checks before submission
# ======================================================================

def check_before_submission(
    df,
):
    if not "prediction" in df.columns:
      raise ValueError(f"The submitted df does not contain a 'prediction' column. Make sure to name it accordingly.")
    if not len(df)==5084:
      raise ValueError(f"The submitted df length ({len(df)}) does not match the length of the test set ({5084}).")
    if not np.all([p in [0,1] for p in df["prediction"]]):
      raise ValueError(f"The submitted predictions contain unknown labels please make sure to respect the format. Prediction should be integers: 0 or 1.")

    print("✅ Submission DataFrame is good to go!")
    return True

def save_for_submission(
    df,
    group_name:str,
):
  # Never too sure: double-check
  submission_ok = check_before_submission(df)
  if submission_ok:
    try: # try to save with current group name
        df.to_csv(f'{group_name}.csv', mode='x')
        print(f"File saved at: {group_name}.csv")
    except FileExistsError: # if file already exist --> modify name and try again
        if type(group_name.split("-")[-1])==int:
            sub_id = int(group_name.split("-")[-1])
            next_sub_id = sub_id+1
        else:
            next_sub_id = 1
        modified_groupname = f"{group_name}-{next_sub_id}"
        save_for_submission(df, modified_groupname)
  else:
      raise ValueError(f"The submission format is not respected. Please modify your submission df.")

# 🟣 Load and explore the data

## Loading dataset

In [14]:
dataset = load_dataset_from_github("data/canon_challenge/dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 6955
    })
    validation: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 5146
    })
    test: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 5084
    })
})

⚠️ Caution: This time we are working with excerpts from French-language novels (not in English anymore)! It might be worth keeping this in mind when implementing your classifier.

See for example:

In [27]:
print(dataset["train"]["text"][0])

Les uns la blâmèrent, les autres l’approuvèrent ; enfin. beaucoup secouèrent la tête en disant que le premier mari était mort au bout de trois mois, le second au bout de deux mois, le troisième au bout d’un mois, et que, pour ne pas faire mentir le calcul nécrologique, je mourrais probablement, moi, la première nuit de mes noces. Mais la personne sur laquelle le coup porta le plus violemment fut la pauvre Schimindra. Les bontés que j’avais eues pour elle lui avaient fait pendant quelque temps concevoir l’espoir de devenir ma femme. Dans un moment de désespoir, elle m’avoua jusqu’où avait été son ambition ; mais je lui fis promptement et facilement comprendre quelle supériorité avait la belle Vanly-Tching, veuve d’un docteur, veuve d’un mandarin, veuve d’un juge civil, sur elle, qui n’était veuve que d’un singe.


The **task** proposed in this notebook is to **predict the `label`** of this dataset which encodes if a literary piece is considered **part of the "literary canon" or not**.

See below some examples in our training data:

In [35]:
n_print = 10

rdm_ids = np.random.randint(0, len(dataset["train"]), n_print)

for rdm_i in rdm_ids:
  rdm_row = dataset['train'][int(rdm_i)]
  print("---------")
  print(f"LABEL    : {rdm_row['label']} ( = {dataset['train'].features['label'].int2str(rdm_row['label'])})")
  print(f"TEXT     : {rdm_row['text']}")
  print(f"METADATA : Author: {rdm_row['author']} | Author Gender: {rdm_row['author_gender']} (= {dataset['train'].features['author_gender'].int2str(rdm_row['author_gender'])}) | Book title: {rdm_row['book_title']} | Publication year: {rdm_row['publication_year']}")

---------
LABEL    : 0 ( = canon)
TEXT     : Le père Ortolan, méridional à tête fine, jurisconsulte de renom, était aussi poète à ses heures. Il avait publié les Enfantines et tout en jurant ne jamais écrire que pour le jeune âge, il ne dédaignait pas à l'endroit de ses vers l'approbation des grandes personnes. Aussi ses soirées, très suivies par les indigènes des quartiers savants, offraient-elles un agréable et original mélange de jolies femmes, de professeurs et d'avocats, de gens doctes et de poètes. C'est comme poète qu'on m'invitait. Parmi les jeunes et antiques célébrités que je vis passer là dans le brouillard d'or des premiers éblouissements, vint un soir Emile Ollivier.
METADATA : Author: Daudet Alphonse | Author Gender: 1 (= male) | Book title: Souvenirs d un homme de lettres. | Publication year: 1888
---------
LABEL    : 1 ( = non-canon)
TEXT     :  — Tu pourras très bien déjeuner demain avec le reste du poulet, dit-elle en le serrant dans le garde-manger, où il alla rejoin

ℹ️ Note: you can train your classifiers using the training set, and evaluate in (/optimize hyperparameters) using the validation split. But you cannot use the test set. Indeed, the `label` (and only the labels, no other metadata) are kept secret untill submission! See:

In [26]:
set(list(dataset["test"]["label"])) # only 'None'!

{None}

## Explore the data

Ussually, before diving into the training part, it can be great to familiarize a bit with the data. This can sometimes even inform the design of the ensuing classifier, or warn on potential difficulties that may arise...

In [ ]:
# Tip: if more comfortable with pandas, you can convert each split of the data to pandas DataFrame with, e.g.:
# df_train = dataset["train"].to_pandas()

# 🟠 Implement your classifier

Now it is time to devise your classifier! You can be creative or conservative here. Try to think for instance on the way to use the data, the type of algorithm that you want to use, etc. (-> no bad answers here!).

Start by implementing the architecture and training components of your classifier. Train and validate it on the `"train"` and `"validation"` splits, and finally get the predictions on the `"test"` set to submit your results.

Feel free to re-use code from the [companion tutorial](https://github.com/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/3_supervised/Tutorial_3_SFT.ipynb): be it fine-tuning example or obtaining documents representations (BoW, TF-IDF, dense embeddings, ...) and classifying them in the representation space.

## Train/Validate your classifier

## Run your classifier on the test set

# 🟡 Upload your Submission!


Your time to shine: store your predictions on the test set in a `pandas.DataFrame` into a column named `"prediction"`.

Check that your DataFrame complies with the necessary format for submission, if not, please modify it so that it respects the guidelines.

If it is compliant the format, then you can save the prediction DataFrame into `csv` format, named with the name of your submission, and uploaed it on the leaderboard submission platform at: https://leaderboard-performance-fairness.streamlit.app/.

In [ ]:
# # TODO: uncomment the following lines and store your predictions in a pandas DataFrame

# df_submission = pd.DataFrame()
# df_submission["prediction"] = [] # YOUR PREDICTIONS HERE!

In [44]:
check_before_submission(df_submission)

✅ Submission DataFrame is good to go!


True

If you passed the quick format check: great! You can now save your predictions and upload it to the leaderboard to see how you did on the *hidden* test set!

In [11]:
save_for_submission(
    df_submission,
    group_name=, # YOUR GROUP NAME HERE
)

✅ Submission DataFrame is good to go!
File saved at: trial_submission.csv


*Bravo!* You can now download your submission-ready `csv` file and upload it on the [leaderboard](https://leaderboard-performance-fairness.streamlit.app/) to see how you did on the `test` set and examine your performance and fairness scores!


## *What just happened...?*

Now, take a little bit of time to think about this experiment. Try to think further: what are the potential causes of bias? How could one try to mitigate it? What does training a model on such task even means? What does it *really* predict when inferring (non-)canonicity? ...